# Logical Planning Agent

States are immutable sets of propositions. Each action has positive/negative
preconditions and add/delete effects. Logic determines which actions are applicable
and how a successor state is produced; breadth-first search finds a shortest valid
action sequence that entails the goal.

In [1]:
from collections import deque
from dataclasses import dataclass


@dataclass(frozen=True)
class Action:
    name: str
    positive_preconditions: frozenset[str]
    negative_preconditions: frozenset[str]
    positive_effects: frozenset[str]
    negative_effects: frozenset[str]

    def applicable(self, state):
        return (
            self.positive_preconditions <= state
            and self.negative_preconditions.isdisjoint(state)
        )

    def apply(self, state):
        if not self.applicable(state):
            raise ValueError(f"Action {self.name} is not applicable")
        return frozenset((state - self.negative_effects) | self.positive_effects)


def at_robot(location):
    return f"At(Robot,{location})"


def at_package(location):
    return f"At(Package,{location})"


HOLDING = "Holding(Package)"
LOCATIONS = ("A", "B", "C")
CONNECTIONS = (("A", "B"), ("B", "A"), ("B", "C"), ("C", "B"))


def build_actions(include_pickup=True):
    actions = []
    for source, target in CONNECTIONS:
        actions.append(
            Action(
                f"Move({source},{target})",
                frozenset({at_robot(source)}),
                frozenset(),
                frozenset({at_robot(target)}),
                frozenset({at_robot(source)}),
            )
        )
    for location in LOCATIONS:
        if include_pickup:
            actions.append(
                Action(
                    f"PickUp(Package,{location})",
                    frozenset({at_robot(location), at_package(location)}),
                    frozenset({HOLDING}),
                    frozenset({HOLDING}),
                    frozenset({at_package(location)}),
                )
            )
        actions.append(
            Action(
                f"Drop(Package,{location})",
                frozenset({at_robot(location), HOLDING}),
                frozenset(),
                frozenset({at_package(location)}),
                frozenset({HOLDING}),
            )
        )
    return actions


def breadth_first_plan(initial_state, goal, actions):
    initial_state, goal = frozenset(initial_state), frozenset(goal)
    frontier = deque([initial_state])
    parent = {initial_state: (None, None)}

    while frontier:
        state = frontier.popleft()
        if goal <= state:
            plan = []
            while parent[state][0] is not None:
                previous, action = parent[state]
                plan.append(action)
                state = previous
            return list(reversed(plan))

        for action in actions:
            if action.applicable(state):
                successor = action.apply(state)
                if successor not in parent:
                    parent[successor] = (state, action)
                    frontier.append(successor)
    return None


def execute_and_verify(initial_state, goal, plan):
    state = frozenset(initial_state)
    trace = [state]
    for action in plan:
        assert action.applicable(state), f"Invalid step: {action.name}"
        state = action.apply(state)
        trace.append(state)
    assert frozenset(goal) <= state, "The final state does not satisfy the goal"
    return trace


def show_trace(trace, plan):
    print("S0:", ", ".join(sorted(trace[0])))
    for index, (action, state) in enumerate(zip(plan, trace[1:]), start=1):
        print(f"  -- {action.name} -->")
        print(f"S{index}:", ", ".join(sorted(state)))

## Test A: solvable warehouse problem

In [2]:
INITIAL = frozenset({at_robot("A"), at_package("A")})
GOAL = frozenset({at_package("C")})
ACTIONS = build_actions()

plan = breadth_first_plan(INITIAL, GOAL, ACTIONS)
assert plan is not None
trace = execute_and_verify(INITIAL, GOAL, plan)

print("Plan found:", [action.name for action in plan])
show_trace(trace, plan)

Plan found: ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)']
S0: At(Package,A), At(Robot,A)
  -- PickUp(Package,A) -->
S1: At(Robot,A), Holding(Package)
  -- Move(A,B) -->
S2: At(Robot,B), Holding(Package)
  -- Move(B,C) -->
S3: At(Robot,C), Holding(Package)
  -- Drop(Package,C) -->
S4: At(Package,C), At(Robot,C)


## Tests B and C: impossible goal and irrelevant movement

In [3]:
# Test B: without PickUp actions the package can never leave A.
impossible_plan = breadth_first_plan(INITIAL, GOAL, build_actions(include_pickup=False))
assert impossible_plan is None
print("Test B:", "No plan found")

# Test C: moving only the robot to C must not satisfy At(Package,C).
action_by_name = {action.name: action for action in ACTIONS}
movement_only = [action_by_name["Move(A,B)"], action_by_name["Move(B,C)"]]
movement_state = INITIAL
for action in movement_only:
    movement_state = action.apply(movement_state)
assert at_robot("C") in movement_state
assert not GOAL <= movement_state
print("Test C final state:", sorted(movement_state))
print("Robot at C is not confused with package at C:", not GOAL <= movement_state)

Test B: No plan found
Test C final state: ['At(Package,A)', 'At(Robot,C)']
Robot at C is not confused with package at C: True


## Additional precondition and plan-validity checks

In [4]:
pickup_a = action_by_name["PickUp(Package,A)"]
drop_c = action_by_name["Drop(Package,C)"]
assert pickup_a.applicable(INITIAL)
assert not drop_c.applicable(INITIAL)

# Every state must keep the robot in exactly one location. The package must be either
# at one location or held, never both.
for state in trace:
    assert sum(at_robot(loc) in state for loc in LOCATIONS) == 1
    package_locations = sum(at_package(loc) in state for loc in LOCATIONS)
    assert package_locations + int(HOLDING in state) == 1

print("Precondition checks and all state invariants passed.")

Precondition checks and all state invariants passed.
